# VisualSync on Google Colab

Beat-synced dance video builder with a web interface.

**Before you start:** *Runtime → Change runtime type →* **T4 GPU** or **L4 GPU**.
A100 and H100 have no video encoder (NVENC), so they render on the CPU — it works, just slower.

Run the cells in order. The last one prints a link that opens the app (it also works on a phone).

In [ ]:
#@title 1. Check the GPU and ffmpeg
!nvidia-smi --query-gpu=name,driver_version --format=csv,noheader || echo "No NVIDIA GPU - rendering will use the CPU"
!which ffmpeg > /dev/null || (apt-get -qq update && apt-get -qq install -y ffmpeg > /dev/null)
!ffmpeg -hide_banner -version | head -1

In [ ]:
#@title 2. Google Drive (recommended for long mixes)
#@markdown Mounts your Drive and creates `MyDrive/VisualSync/video`, `audio` and `output`.
#@markdown Put the dancer clips in `video` and the DJ Mix in `audio`, then choose
#@markdown **Folders** in the app — much faster than uploading through the browser,
#@markdown and finished videos are saved straight to `output` (Colab's own disk is wiped
#@markdown when the session ends).
USE_DRIVE = True #@param {type:"boolean"}
#@markdown **SEND_TO_YOUTUBE** also signs you in to Google so the app can hand finished
#@markdown videos to your IFTTT Applet (via the *YouTube Queue* sheet). See cell 5.
SEND_TO_YOUTUBE = True #@param {type:"boolean"}
if USE_DRIVE:
    import os
    from google.colab import drive
    drive.mount("/content/drive")
    for sub in ("video", "audio", "output"):
        os.makedirs(f"/content/drive/MyDrive/VisualSync/{sub}", exist_ok=True)
    print("Ready: /content/drive/MyDrive/VisualSync/{video,audio,output}")
if SEND_TO_YOUTUBE:
    from google.colab import auth
    auth.authenticate_user()
    print("Signed in: finished videos can be sent to YouTube.")

In [ ]:
#@title 3. Get VisualSync and install Gradio
REPO = "https://github.com/enoky/VisualSync" #@param {type:"string"}
BRANCH = "main" #@param {type:"string"}
import os
if os.path.isdir("/content/VisualSync/.git"):
    !git -C /content/VisualSync fetch -q origin {BRANCH} && git -C /content/VisualSync checkout -q -B {BRANCH} FETCH_HEAD
else:
    !git clone -q -b {BRANCH} {REPO} /content/VisualSync
!pip install -q -U "gradio>=5" tqdm
print("Installed.")

In [ ]:
#@title 4. Start the app
#@markdown Click the **public URL** (`https://….gradio.live`) that appears below.
#@markdown Keep this cell running while you use the app; stop it to shut the app down.
%cd /content/VisualSync
!python gradio_app.py --share

In [ ]:
#@title 5. Send a finished video to YouTube (via IFTTT)
#@markdown The app can do this for you (*Send to YouTube* section). Use this cell for a video
#@markdown you rendered earlier — stop cell 4 first, or run it in a new session after cells 2–3.
#@markdown
#@markdown It shares the video by link, then adds a row to the **YouTube Queue** Google Sheet
#@markdown (A = direct link, B = title, C = description). Your IFTTT Applet uploads it to
#@markdown YouTube as Private, usually within an hour. Nothing passes through your phone.
VIDEO = "/content/drive/MyDrive/VisualSync/output/my_video.mp4" #@param {type:"string"}
TITLE = "" #@param {type:"string"}
DESCRIPTION = "" #@param {type:"string"}
import sys
sys.path.insert(0, "/content/VisualSync")
import youtube_queue
try:
    for msg in youtube_queue.queue_video(VIDEO, TITLE, DESCRIPTION):
        print(msg)
except youtube_queue.QueueError as exc:
    print("Not queued:", exc)

### Tips
- **Render preview** first (30 s by default) to check sync, the downbeat offset and quality.
- The line at the top of the app shows what this runtime can do. *Auto* uses the GPU for
  encoding when NVENC works and falls back to the CPU otherwise.
- Anyone with the gradio.live link can use the app while this notebook runs, so don't share it.
- **YouTube:** after the video is on YouTube (Studio shows it), you can switch its Drive file
  back to *Restricted* — the link is only needed while IFTTT uploads it.